In [14]:
from pathlib import Path
import shutil

from tqdm import tqdm
from PIL import Image, ImageOps, ImageDraw

import numpy as np
from kraken import blla
from kraken.lib import vgsl

# Preprocessing

We copy only the recto side (ending in `o.jpg`, for **o**orkonde) from the original Ghent photo archive to a flat new folder. We convert JPG to PNG.

In [15]:
SRC_DIR = Path("../images/archive-original")
DST_DIR = Path("../images/archive-recto")

assert SRC_DIR.exists(), f"Source folder not found: {SRC_DIR.resolve()}"

# reset destination
if DST_DIR.exists():
    shutil.rmtree(DST_DIR)
    print(f"Removed existing: {DST_DIR.resolve()}")
DST_DIR.mkdir(parents=True, exist_ok=True)
print(f"Created fresh:    {DST_DIR.resolve()}")

matches = [
    p for p in SRC_DIR.rglob("*.jpg")
    if p.is_file() and p.name.lower().endswith("o.jpg")
]

converted = 0
for src in tqdm(matches, desc="Copying -> PNG", unit="file"):
    img = Image.open(src)
    img = ImageOps.exif_transpose(img) # bake in EXIF rotation
    img = img.convert("RGB")
    img.save(DST_DIR / f"{src.stem}.png", "PNG")
    converted += 1

print(f"Matched:   {len(matches)} file(s)")
print(f"Converted: {converted} file(s) into {DST_DIR.resolve()}")

Removed existing: /Users/mikekestemont/Desktop/speld_hooiberg/images/archive-recto
Created fresh:    /Users/mikekestemont/Desktop/speld_hooiberg/images/archive-recto


Copying -> PNG: 100%|██████████| 1408/1408 [02:42<00:00,  8.67file/s]

Matched:   1408 file(s)
Converted: 1408 file(s) into /Users/mikekestemont/Desktop/speld_hooiberg/images/archive-recto


We use Kraken's layout to identify and cut out the main text zone from these images. We store this new crop in a binarized version (Sauvola's method) in a new folder:

In [16]:
CROP_DIR       = Path("../images/cropped")
DEVICE         = "cpu"      # or "cuda:0"
MODEL_PATH     = None       # None -> kraken's bundled blla.mlmodel
BINARIZE       = True
WINDOW_SIZE    = 51         # Sauvola window (odd)
SAUVOLA_K      = 0.3        # Sauvola sensitivity
MIN_SIZE       = 256        # skip crops smaller than this on either side
MAX_BLACK_RATIO = 0.5       # skip if binarized result is >50% black

def sauvola_threshold(a, window_size=51, k=0.3):
    """Per-pixel Sauvola threshold via integral images."""
    if window_size % 2 == 0:
        window_size += 1
    img = a.astype(np.float64)
    R, half = 128.0, window_size // 2
    integ = np.zeros((img.shape[0] + 1, img.shape[1] + 1))
    integ_sq = np.zeros_like(integ)
    np.cumsum(np.cumsum(img, 0), 1, out=integ[1:, 1:])
    np.cumsum(np.cumsum(img ** 2, 0), 1, out=integ_sq[1:, 1:])
    rows, cols = img.shape
    r1 = np.clip(np.arange(rows) - half, 0, rows)[:, None]
    r2 = np.clip(np.arange(rows) + half + 1, 0, rows)[:, None]
    c1 = np.clip(np.arange(cols) - half, 0, cols)[None, :]
    c2 = np.clip(np.arange(cols) + half + 1, 0, cols)[None, :]
    area = (r2 - r1) * (c2 - c1)
    s  = integ[r2, c2] - integ[r2, c1] - integ[r1, c2] + integ[r1, c1]
    sq = integ_sq[r2, c2] - integ_sq[r2, c1] - integ_sq[r1, c2] + integ_sq[r1, c1]
    mean = s / area
    std = np.sqrt(np.clip(sq / area - mean ** 2, 0, None))
    return mean * (1.0 + k * (std / R - 1.0))

def binarize_image(a, window_size=51, k=0.3):
    return np.where(a < sauvola_threshold(a, window_size, k), 0, 255).astype(np.uint8)

def find_main_text_region(seg):
    """Boundary polygon of the largest region by bbox area, or None."""
    all_regions = [r for rl in seg.regions.values() for r in rl]
    if not all_regions:
        return None
    def area(r):
        xs = [p[0] for p in r.boundary]; ys = [p[1] for p in r.boundary]
        return (max(xs) - min(xs)) * (max(ys) - min(ys))
    return max(all_regions, key=area).boundary

def extract_region(image, polygon, out_path):
    """Mask polygon onto white, crop to bbox, optionally binarize, save PNG.
    Returns (saved, reason)."""
    img_a = np.array(image.convert("L"))
    H, W = img_a.shape
    polygon = [(max(0, min(x, W - 1)), max(0, min(y, H - 1))) for x, y in polygon]
    xs = [p[0] for p in polygon]; ys = [p[1] for p in polygon]
    min_x, max_x, min_y, max_y = min(xs), max(xs), min(ys), max(ys)
    if (max_x - min_x) < MIN_SIZE or (max_y - min_y) < MIN_SIZE:
        return False, f"region too small ({max_x-min_x}x{max_y-min_y})"
    mask = Image.new("L", image.size, 0)
    ImageDraw.Draw(mask).polygon(polygon, fill=255)
    out = np.where(np.array(mask) == 255, img_a, 255)[min_y:max_y, min_x:max_x]
    if out.shape[0] < MIN_SIZE or out.shape[1] < MIN_SIZE:
        return False, "crop too small"
    if BINARIZE:
        if out.std() < 1:
            return False, "near-uniform, skipped"
        out = binarize_image(out, WINDOW_SIZE, SAUVOLA_K)
        black = np.count_nonzero(out == 0) / out.size
        if black > MAX_BLACK_RATIO:
            return False, f"too much black ({black:.0%})"
    Image.fromarray(out).save(out_path, "PNG")
    return True, ""

# reset output folder
if CROP_DIR.exists():
    shutil.rmtree(CROP_DIR)
CROP_DIR.mkdir(parents=True, exist_ok=True)

# load segmentation model once (re-run-friendly: del seg_model to force reload)
if "seg_model" not in globals():
    if MODEL_PATH is None:
        import kraken as _k
        MODEL_PATH = str(Path(_k.__file__).parent / "blla.mlmodel")
    seg_model = vgsl.TorchVGSLModel.load_model(MODEL_PATH)
    print(f"Loaded model: {MODEL_PATH}")

image_files = sorted(DST_DIR.glob("*.png"))
saved = skipped = fallbacks = 0

for p in tqdm(image_files, desc="Segmenting", unit="file"):
    try:
        im = Image.open(p)                      # already oriented upstream
        out_path = CROP_DIR / f"{p.stem}.png"
        polygon = None
        try:
            polygon = find_main_text_region(blla.segment(im, model=seg_model, device=DEVICE))
        except Exception as e:
            tqdm.write(f"  ⚠ {p.name}: segmentation failed ({e}), using full image")
        if polygon is None:                     # fallback: whole image
            w, h = im.size
            polygon = [(0, 0), (w, 0), (w, h), (0, h)]
            ok, reason = extract_region(im, polygon, out_path)
            if ok: fallbacks += 1
        else:
            ok, reason = extract_region(im, polygon, out_path)
        if ok:
            saved += 1
        else:
            tqdm.write(f"  ⚠ {p.name}: skipped ({reason})")
            skipped += 1
    except Exception as e:
        tqdm.write(f"  ⚠ {p.name}: {e}")
        skipped += 1

print(f"\nDone. Saved: {saved} ({fallbacks} full-image fallbacks), Skipped: {skipped}")
print(f"Output: {CROP_DIR.resolve()}")

/Users/mikekestemont/miniconda3/envs/bayes/lib/python3.10/site-packages/coremltools/models/model.py:560: RuntimeWarning: You will not be able to run predict() on this Core ML model. Underlying exception message was: Error compiling model: "compiler error: Error reading protobuf spec. validator error: Input MLMultiArray to neural networks must have dimension 1 (vector) or 3 (image-like arrays).".
  _warnings.warn(


Loaded model: /Users/mikekestemont/miniconda3/envs/bayes/lib/python3.10/site-packages/kraken/blla.mlmodel


Segmenting:   9%|▉         | 131/1408 [23:38<4:00:58, 11.32s/file]Polygonizer failed on line 0: TopologyException: side location conflict at 343.69387755102042 835.61224489795916. This can occur if the input geometry is invalid.
Polygonizer failed on line 0: TopologyException: side location conflict at 376.79487179487177 834.79487179487182. This can occur if the input geometry is invalid.
Segmenting:  19%|█▉        | 271/1408 [48:38<3:06:11,  9.83s/file]

  ⚠ 1244o.png: skipped (region too small (658x172))


Segmenting:  20%|██        | 282/1408 [50:39<3:27:36, 11.06s/file]

...

  ⚠ 1312o.png: skipped (too much black (57%))


Segmenting:  60%|█████▉    | 838/1408 [2:33:13<1:44:30, 11.00s/file]Polygonizer failed on line 0: TopologyException: side location conflict at 2455.0100334448161 1030.0033444816054. This can occur if the input geometry is invalid.
Polygonizer failed on line 0: TopologyException: side location conflict at 735.6480686695279 1540.37339055794. This can occur if the input geometry is invalid.
Segmenting:  66%|██████▌   | 925/1408 [2:49:39<1:26:46, 10.78s/file]Polygonizer failed on line 0: TopologyException: side location conflict at 581.18181818181813 1582. This can occur if the input geometry is invalid.
Polygonizer failed on line 0: TopologyException: side location conflict at 738.5 683.75. This can occur if the input geometry is invalid.
Polygonizer failed on line 0: TopologyException: side location conflict at 738.11111111111109 683.39506172839504. This can occur if the input geometry is invalid.
Segmenting:  73%|███████▎  | 1028/1408 [3:08:47<1:11:42, 11.32s/file]Polygonizer failed on 

  ⚠ 718o.png: skipped (region too small (1459x222))


Segmenting: 100%|██████████| 1408/1408 [4:19:57<00:00, 11.08s/file]


Done. Saved: 1392 (0 full-image fallbacks), Skipped: 16
Output: /Users/mikekestemont/Desktop/speld_hooiberg/images/cropped


In [17]:
ADDITIONAL_DIR = Path("../images/additional")
assert ADDITIONAL_DIR.exists(), f"Not found: {ADDITIONAL_DIR.resolve()}"
assert CROP_DIR.exists(), "Run the segmentation cell first (CROP_DIR missing)."

add_files = sorted(
    p for p in ADDITIONAL_DIR.glob("*")
    if p.is_file() and p.suffix.lower() == ".jpg"
)

added = 0
for p in tqdm(add_files, desc="Adding additional -> PNG", unit="file"):
    try:
        im = ImageOps.exif_transpose(Image.open(p))   # raw jpgs may carry orientation
        arr = np.array(im.convert("L"))
        if BINARIZE:
            arr = binarize_image(arr, WINDOW_SIZE, SAUVOLA_K)
        Image.fromarray(arr).save(CROP_DIR / f"{p.stem}.png", "PNG")
        added += 1
    except Exception as e:
        tqdm.write(f"  ⚠ {p.name}: {e}")

print(f"Added {added} of {len(add_files)} additional image(s) into {CROP_DIR.resolve()}")

Adding additional -> PNG: 100%|██████████| 6/6 [00:02<00:00,  2.20file/s]

Added 6 of 6 additional image(s) into /Users/mikekestemont/Desktop/speld_hooiberg/images/cropped
